# 23. Baseline — GPT-4o

컬럼쌍마다 테이블·컬럼명, 프로파일링 통계(행 수, 고유값 수·비율, 결측 비율), 샘플 값을 프롬프트로 주고
GPT-4o(temperature 0)가 돌려준 `fk_probability` 를 점수로 삼아 다른 모델과 같은 지표로 평가한다.
아래 `API_KEY` 에 본인의 OpenAI API 키를 넣어 실행한다 (키를 커밋하지 않도록 주의).

- 출력: `workspace/gpt4o_ranking_metrics_<db>.json`

In [ ]:
import json
from pathlib import Path
from itertools import combinations

import pandas as pd
from tqdm import tqdm
from openai import OpenAI

# ==========================================
# 설정
# ==========================================

API_KEY = "open AI  key 입력"

DATASET_DIR = Path("./dataset")

MODEL = "gpt-4o"

THRESHOLD = 0.7

MAX_SAMPLE_VALUES = 30

OUTPUT_FILE = "gpt4o_fk_results.csv"

# ==========================================
# OpenAI Client
# ==========================================

client = OpenAI(api_key=API_KEY)



In [ ]:
# ==========================================
# Prompt
# ==========================================

PROMPT_TEMPLATE = """
You are a database schema matching expert.

Determine whether the following two columns are likely to participate in a Primary Key–Foreign Key (PK-FK) relationship.

A PK-FK relationship typically exhibits:

- Inclusion of values from one column within the other
- High uniqueness on the referenced column
- Consistent identifier patterns
- Compatible value domains

Consider both sample values and column names when available.

Do not rely solely on column names.

Return ONLY valid JSON.

{{
  "fk_probability": 0.00
}}

Column A:
Name: {colA}

Row Count: {row_count_A}
Unique Count: {unique_count_A}
Unique Ratio: {unique_ratio_A}
Null Ratio: {null_ratio_A}

Samples:
{samplesA}

Column B:
Name: {colB}

Row Count: {row_count_B}
Unique Count: {unique_count_B}
Unique Ratio: {unique_ratio_B}
Null Ratio: {null_ratio_B}

Samples:
{samplesB}
"""


In [ ]:
# ==========================================
# CSV Load
# ==========================================

def load_tables(dataset_dir):

    tables = {}

    for csv_file in dataset_dir.glob("*.csv"):

        try:

            df = pd.read_csv(csv_file)

            tables[csv_file.stem] = df

            print(f"Loaded: {csv_file.name}")

        except Exception as e:

            print(f"Skip: {csv_file.name} ({e})")

    return tables


In [ ]:
# ==========================================
# Column Extraction
# ==========================================
import random
def build_columns(tables):

    columns = []

    for table_name, df in tables.items():

        for col in df.columns:

            series = df[col]

            values = (
                series
                .dropna()
                .astype(str)
                .tolist()
            )

            # samples = values[:MAX_SAMPLE_VALUES]
            samples = random.sample(values, min(MAX_SAMPLE_VALUES, len(values)))

            row_count = len(series)

            null_count = int(series.isna().sum())

            unique_count = int(series.nunique(dropna=True))

            null_ratio = (
                null_count / row_count
                if row_count > 0
                else 0.0
            )

            unique_ratio = (
                unique_count / row_count
                if row_count > 0
                else 0.0
            )

            columns.append({
                "table": table_name,
                "column": col,

                "samples": samples,

                "row_count": row_count,
                "null_count": null_count,
                "null_ratio": round(null_ratio, 4),

                "unique_count": unique_count,
                "unique_ratio": round(unique_ratio, 4)
            })

    return columns

In [ ]:
# ==========================================
# GPT Query
# ==========================================
import re
import time

import json
import time


import json
import time


def query_gpt(colA, colB, max_retry=5):

    prompt = PROMPT_TEMPLATE.format(

        colA=f"{colA['table']}.{colA['column']}",
        colB=f"{colB['table']}.{colB['column']}",

        row_count_A=colA["row_count"],
        unique_count_A=colA["unique_count"],
        unique_ratio_A=colA["unique_ratio"],
        null_ratio_A=colA["null_ratio"],

        row_count_B=colB["row_count"],
        unique_count_B=colB["unique_count"],
        unique_ratio_B=colB["unique_ratio"],
        null_ratio_B=colB["null_ratio"],

        samplesA="\n".join(colA["samples"]),
        samplesB="\n".join(colB["samples"])
    )

    for attempt in range(max_retry):

        try:

            response = client.responses.create(
                model=MODEL,
                temperature=0,
                input=prompt
            )

            text = response.output_text

            if text is None:
                raise ValueError("Empty response")

            text = (
                text
                .replace("```json", "")
                .replace("```", "")
                .strip()
            )

            result = json.loads(text)

            prob = float(
                result["fk_probability"]
            )

            prob = max(
                0.0,
                min(1.0, prob)
            )

            pred = int(
                prob >= THRESHOLD
            )

            return prob, pred

        except Exception as e:

            print(
                f"[ERROR] "
                f"{colA['column']} "
                f"<-> "
                f"{colB['column']}"
            )

            print(str(e))

            if "429" in str(e):

                wait_time = 5 * (attempt + 1)

                print(
                    f"Retry after "
                    f"{wait_time}s"
                )

                time.sleep(wait_time)

                continue

            if attempt == max_retry - 1:

                return 0.0, 0

            time.sleep(1)

    return 0.0, 0

In [ ]:
def process_pair(pair):

    colA, colB = pair

    prob, pred = query_gpt(colA, colB)

    return {
        "tableA": colA["table"],
        "columnA": colA["column"],
        "tableB": colB["table"],
        "columnB": colB["column"],
        "fk_probability": prob,
        "prediction": pred
    }

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed

TEST_ROOT = '../workspace/dataset_mini/bird_data_tables'
TARGET_DBS = 'california_schools_mask'
# TARGET_DBS = ['toxicology', 'california_schools', 'superhero', 'thrombosis_prediction']

DATASET_DIR  = Path(TEST_ROOT) / TARGET_DBS

tables = load_tables(DATASET_DIR)

columns = build_columns(tables)

print(f"\nColumns: {len(columns)}")

pairs = list(combinations(columns, 2))

print(f"Pairs: {len(pairs)}")

results = []

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = [
        executor.submit(process_pair, pair)
        for pair in pairs
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        results.append(
            future.result()
        )


In [ ]:
result_df = pd.DataFrame(results)

result_df.sort_values(
    by="fk_probability",
    ascending=False,
    inplace=True
)
OUTPUT_FILE = 'gpt4o_fk_results_california_schools_mask.csv'
result_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print()
print("Saved:", OUTPUT_FILE)

In [ ]:
import json
import pandas as pd

RESULTS_CSV = "../workspace/gpt4o_fk_results_superhero_mask.csv"
GROUND_TRUTH_JSON = "../workspace/dataset_mini/bird_data_tables/superhero_mask/ground_truth.json"

df_pred = pd.read_csv(RESULTS_CSV)

with open(GROUND_TRUTH_JSON) as f:
    gt_pairs_raw = json.load(f)

def to_key(a, b):
    return frozenset({a, b})

truth_set = {to_key(a, b) for a, b in gt_pairs_raw}

all_pairs = [
    to_key(f"{row.tableA}.{row.columnA}", f"{row.tableB}.{row.columnB}")
    for row in df_pred.itertuples(index=False)
]
pred_pos_set = {
    to_key(f"{row.tableA}.{row.columnA}", f"{row.tableB}.{row.columnB}")
    for row in df_pred.itertuples(index=False)
    # if int(row.prediction) == 1
    if row.fk_probability >= 0.5
}

evaluated_truth = truth_set & set(all_pairs)
missing_truth = truth_set - set(all_pairs)

TP = len(pred_pos_set & evaluated_truth)
FP = len(pred_pos_set - evaluated_truth)
FN = len(evaluated_truth - pred_pos_set)
TN = len(all_pairs) - TP - FP - FN

print(f"Total pairs in CSV : {len(all_pairs)}")
print(f"Ground truth pairs : {len(truth_set)}")
if missing_truth:
    print(f"GT pairs not in CSV: {len(missing_truth)} (excluded from FN)")
    for p in missing_truth:
        print(f"  - {sorted(p)}")
print()
print(f"TP: {TP}")
print(f"FP: {FP}")
print(f"TN: {TN}")
print(f"FN: {FN}")

precision = TP / (TP + FP) if (TP + FP) else 0.0
recall    = TP / (TP + FN) if (TP + FN) else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
print()
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

RESULTS_DIR = "../workspace"
GT_ROOT = "../workspace/dataset_mini/bird_data_tables"
RESULT_SUFFIX = ""  # "_mask" 또는 ""
SAVE_JSON_FOR_TOTAL_RESUL = True  # total_resul.ipynb 호환 JSON 저장

DBS = ["toxicology", "california_schools", "superhero", "thrombosis_prediction"]

def to_key(a, b):
    return frozenset({a, b})

def compute_mrr(y_true, y_score):
    yt = np.asarray(y_true)
    ys = np.asarray(y_score)
    if yt.sum() == 0:
        return 0.0
    order = np.argsort(-ys, kind="stable")
    ranked_truth = yt[order]
    rrs = [1.0 / (i + 1) for i, v in enumerate(ranked_truth) if v == 1]
    return float(np.mean(rrs)) if rrs else 0.0

def recall_at_topn(y_true, y_score, n):
    if n <= 0:
        return 0.0
    order = np.argsort(-y_score, kind="stable")
    topn_idx = order[:n]
    denom = int(np.asarray(y_true).sum())
    if denom == 0:
        return 0.0
    return float(np.asarray(y_true)[topn_idx].sum()) / denom

per_db_results = []

for db in DBS:
    csv_path = f"{RESULTS_DIR}/gpt4o_fk_results_{db}{RESULT_SUFFIX}.csv"
    gt_path = f"{GT_ROOT}/{db}/ground_truth.json"

    df = pd.read_csv(csv_path)
    with open(gt_path) as f:
        gt_raw = json.load(f)

    truth = {to_key(a, b) for a, b in gt_raw}

    csv_keys = [
        to_key(f"{row.tableA}.{row.columnA}", f"{row.tableB}.{row.columnB}")
        for row in df.itertuples(index=False)
    ]
    y_score = df["fk_probability"].to_numpy()
    y_true = np.array([1 if k in truth else 0 for k in csv_keys])

    n_gt_total = len(truth)
    n_gt_in_csv = int(y_true.sum())
    K = n_gt_in_csv

    pr_auc = average_precision_score(y_true, y_score) if n_gt_in_csv > 0 else 0.0
    recall_at_k = recall_at_topn(y_true, y_score, K)
    recall_at_3k = recall_at_topn(y_true, y_score, 3 * K)
    mrr = compute_mrr(y_true, y_score)

    per_db_results.append({
        "db": db,
        "n_pairs": len(df),
        "n_gt": n_gt_total,
        "n_gt_in_csv": n_gt_in_csv,
        "pr_auc": pr_auc,
        "recall_at_K": recall_at_k,
        "recall_at_3K": recall_at_3k,
        "mrr": mrr,
    })

    if SAVE_JSON_FOR_TOTAL_RESUL:
        out_path = f"{RESULTS_DIR}/gpt4o_ranking_metrics_{db}.json"
        payload = {
            "model": "GPT-4o",
            "db": db,
            "pr_auc": float(pr_auc),
            "recall_at_K": float(recall_at_k),
            "recall_at_3K": float(recall_at_3k),
            "mrr": float(mrr),
            "n_gt": int(n_gt_total),
            "total_pairs": int(len(df)),
            "y_true": y_true.tolist(),
            "y_score": y_score.tolist(),
        }
        with open(out_path, "w") as f:
            json.dump(payload, f)
        print(f"saved: {out_path}")

res_df = pd.DataFrame(per_db_results)

print("\nPer-DB metrics:")
print(res_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print()
print("Macro average across DBs:")
print(f"  PR-AUC      : {res_df['pr_auc'].mean():.4f}")
print(f"  Recall@K    : {res_df['recall_at_K'].mean():.4f}")
print(f"  Recall@3K   : {res_df['recall_at_3K'].mean():.4f}")
print(f"  MRR         : {res_df['mrr'].mean():.4f}")